# JEPA For Time Series

> I guess so!

In [ ]:
#| default_exp jepa

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, math, torch.nn.functional as F, torch.nn as nn, copy, numpy as np, lightning.pytorch as pl, warnings

from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts
from torch.nn.attention import SDPBackend, sdpa_kernel
from rotary_embedding_torch import RotaryEmbedding, apply_rotary_emb
from physiojepa.layers import MultiHeadAttention, MLP, Patch, tAPE, PositionalEncoding, get_activation_fn
from physiojepa.tokenizers import TS_Tokenizer, TS_Tokenizer_Complex, InceptionTokenizer, PatchEncoder
from physiojepa.utils import trunc_normal_
from physiojepa.patchtst import TSTBlock

## Torch

In [ ]:
#| export
class PositionAwareMultiHeadAttention(nn.Module):
    """Native-JEPA attention that applies RoPE using original patch positions."""
    def __init__(self, dim, num_heads=8, qkv_bias=False, qk_scale=None,
                 attn_drop=0., proj_drop=0., rotary_pes=False):
        super().__init__()
        if dim % num_heads:
            raise ValueError(f"dim ({dim}) must be divisible by num_heads ({num_heads})")
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = qk_scale or self.head_dim ** -0.5
        self.W_Q = nn.Linear(dim, dim, bias=qkv_bias)
        self.W_K = nn.Linear(dim, dim, bias=qkv_bias)
        self.W_V = nn.Linear(dim, dim, bias=qkv_bias)
        self.attn_drop_prob = attn_drop
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.rotary_pes = rotary_pes
        if rotary_pes:
            self.rotary_embed = RotaryEmbedding(
                dim=self.head_dim, freqs_for="lang", theta=10000,
                learned_freq=False, seq_before_head_dim=False,
                use_xpos=False, cache_max_seq_len=29000,
            )

    def _apply_rotary(self, tensor, positions):
        if positions is None:
            positions = torch.arange(tensor.shape[-2], device=tensor.device)
        positions = positions.to(device=tensor.device)
        frequencies = self.rotary_embed(positions)
        if frequencies.ndim == 3:
            frequencies = frequencies.unsqueeze(1)
        return apply_rotary_emb(
            frequencies, tensor, seq_dim=-2, freqs_seq_dim=-2
        )

    def forward(self, x, key=None, value=None, mask=None, positions=None,
                key_positions=None):
        key = x if key is None else key
        value = x if value is None else value
        q = self.W_Q(x).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        k = self.W_K(key).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        v = self.W_V(value).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        if self.rotary_pes:
            q = self._apply_rotary(q, positions)
            k = self._apply_rotary(k, positions if key_positions is None else key_positions)
        attention_dropout = self.attn_drop_prob if self.training else 0.0
        with sdpa_kernel(
            [SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH],
            set_priority=True,
        ):
            output = F.scaled_dot_product_attention(
                q, k, v, attn_mask=mask, dropout_p=attention_dropout,
                is_causal=False, scale=self.scale,
            )
        output = output.transpose(1, 2).flatten(-2)
        return self.proj_drop(self.proj(output))


class PositionAwareTSTBlock(nn.Module):
    """TST block with explicit original positions for native-JEPA RoPE."""
    def __init__(self, d_model, n_heads, d_ff=256, attn_dropout=0,
                 dropout=0., bias=True, activation="gelu", pre_norm=False,
                 rotary_pes=False):
        super().__init__()
        self.self_attn = PositionAwareMultiHeadAttention(
            dim=d_model, num_heads=n_heads, qkv_bias=bias,
            attn_drop=attn_dropout, proj_drop=dropout,
            rotary_pes=rotary_pes,
        )
        self.dropout_attn = nn.Dropout(dropout)
        self.norm_attn = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=bias), get_activation_fn(activation),
            nn.Dropout(dropout), nn.Linear(d_ff, d_model, bias=bias),
        )
        self.dropout_ffn = nn.Dropout(dropout)
        self.norm_ffn = nn.LayerNorm(d_model)
        self.pre_norm = pre_norm

    def forward(self, src, mask=None, positions=None):
        if self.pre_norm:
            src = self.norm_attn(src)
        src2 = self.self_attn(src, mask=mask, positions=positions)
        src = src + self.dropout_attn(src2)
        if not self.pre_norm:
            src = self.norm_attn(src)
        if self.pre_norm:
            src = self.norm_ffn(src)
        src2 = self.ff(src)
        src = src + self.dropout_ffn(src2)
        if not self.pre_norm:
            src = self.norm_ffn(src)
        return src


class JEPABlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        mlp_ratio=4.0,
        qkv_bias=False,
        qk_scale=None,
        drop=0.,
        attn_drop=0.,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm,
        rotary_pes=False
    ):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = PositionAwareMultiHeadAttention(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            qk_scale=qk_scale,
            attn_drop=attn_drop,
            proj_drop=drop,
            rotary_pes=rotary_pes
            )

        self.norm2 = norm_layer(dim)
        self.rotary_pes = rotary_pes 
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = MLP(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            act_layer=act_layer,
            drop=drop)

    def forward(self, x, mask=None, positions=None):
        x = self.norm1(x)
        y = self.attn(x, key=x, value=x, mask=mask, positions=positions)
        x = x + y
        x = x + self.mlp(self.norm2(x))
        return x

def apply_masks(x, masks):
    all_x = []
    for m, x_i in zip(masks, x):
        mask_keep = m.unsqueeze(-1).repeat(1, x_i.size(-1))
        x_i_masked = torch.gather(x_i, dim=0, index=mask_keep)
        assert x_i_masked.shape[0] == mask_keep.shape[0], "The number of masked tokens does not match the number of masked positions"
        all_x.append(x_i_masked)
    return torch.stack(all_x, dim=0)

def apply_position_masks(positions, masks):
    """Gather original patch positions using one index row per batch item."""
    if positions.ndim != 2 or masks.ndim != 2:
        raise ValueError("positions and masks must both have shape [batch, patches]")
    if positions.shape[0] != masks.shape[0]:
        raise ValueError("positions and masks must have the same batch dimension")
    return torch.gather(positions, dim=1, index=masks.to(positions.device))

def representation_channel_stats(x, max_vectors=1024):
    """Return bounded per-channel mean, std, and pairwise cosine diagnostics."""
    batch_size, channels, num_patches, d_model = x.shape
    sample_size = min(num_patches, max(1, max_vectors // max(batch_size, 1)))
    indices = torch.randperm(num_patches, device=x.device)[:sample_size]
    sampled = x[:, :, indices, :].permute(0, 2, 1, 3)
    flattened = sampled.reshape(batch_size * sample_size, channels, d_model)
    channel_stats = []
    for channel_index in range(channels):
        values = flattened[:, channel_index, :]
        normalized = F.normalize(values, dim=1)
        similarity = normalized @ normalized.T
        if similarity.shape[0] > 1:
            off_diagonal = ~torch.eye(
                similarity.shape[0], dtype=torch.bool, device=similarity.device
            )
            cosine_similarity = similarity[off_diagonal].mean().item()
        else:
            cosine_similarity = 1.0
        channel_stats.append(
            (values.mean().item(), values.std().item(), cosine_similarity)
        )
    return channel_stats


def create_masks(x, patch_size, patch_stride, context_mask_range, target_mask_range, melt_channels_to_batch=False):
    """Create fixed-width masks from one ratio draw shared by the batch."""
    for name, ratio_range in (("context", context_mask_range), ("target", target_mask_range)):
        if len(ratio_range) != 2 or not 0 < ratio_range[0] <= ratio_range[1] < 1:
            raise ValueError(f"Invalid {name} mask range: {ratio_range}")
    batch_size = 0
    channel_size = x.size(1)
    sample_n_patches = []
    for x_i in x:
        num_patches = int((max(x_i.size(-1), patch_size)-patch_size) // patch_stride + 1)
        if ((x_i.size(-1)-patch_size) % patch_stride != 0):
            num_patches += 1
        if melt_channels_to_batch:
            added_batches = channel_size
            batch_size += added_batches
            sample_n_patches.extend(num_patches for i in range(added_batches))
        else:
            batch_size += 1
            sample_n_patches.append(num_patches)
    
    if not sample_n_patches or len(set(sample_n_patches)) != 1:
        raise ValueError("All samples must produce the same number of patches")
    n_patch = sample_n_patches[0]
    target_ratio = target_mask_range[0] + torch.rand(1).item() * (target_mask_range[1] - target_mask_range[0])
    context_ratio = context_mask_range[0] + torch.rand(1).item() * (context_mask_range[1] - context_mask_range[0])
    num_target = max(1, int(n_patch * target_ratio))
    remaining = n_patch - num_target
    num_context = max(1, int(remaining * context_ratio))
    if num_target + num_context > n_patch:
        raise ValueError("Target and context masks exceed the patch count")

    permutations = torch.stack([torch.randperm(n_patch) for _ in range(batch_size)])
    mask_indices = permutations[:, :num_target]
    non_mask_indices = permutations[:, num_target:num_target + num_context]
    return mask_indices, non_mask_indices

In [ ]:
def test_apply_masks_preserves_effective_batch():
    x = torch.arange(6 * 5 * 2).reshape(6, 5, 2)
    masks = torch.tensor([[0, 2, 4], [1, 2, 3], [0, 1, 4], [2, 3, 4], [0, 3, 4], [1, 3, 4]])
    masked = apply_masks(x, masks)
    assert masked.shape == (6, 3, 2)
    for batch_idx in range(x.shape[0]):
        assert torch.equal(masked[batch_idx], x[batch_idx, masks[batch_idx]])

def test_mask_ratios_do_not_collapse_across_the_effective_batch():
    torch.manual_seed(12)
    x = torch.empty(128, 3, 1800)
    observed = []
    for _ in range(12):
        targets, contexts = create_masks(x, 10, 10, (0.1, 0.4), (0.1, 0.3), True)
        target_ratio = targets.shape[1] / 180
        context_ratio = contexts.shape[1] / (180 - targets.shape[1])
        assert 0.1 - 1 / 180 <= target_ratio <= 0.3
        assert 0.1 - 1 / 180 <= context_ratio <= 0.4
        assert targets.shape[0] == contexts.shape[0] == 128 * 3
        for target_row, context_row in zip(targets, contexts):
            assert set(target_row.tolist()).isdisjoint(context_row.tolist())
        observed.append((targets.shape[1], contexts.shape[1]))
    assert len(set(observed)) > 1

def test_position_aware_rotary_attention_is_permutation_equivariant():
    torch.manual_seed(12)
    attention = PositionAwareMultiHeadAttention(16, 4, rotary_pes=True).eval()
    x = torch.randn(2, 31, 16)
    positions = torch.arange(31).expand(2, -1)
    permutation = torch.randperm(31)
    inverse = torch.argsort(permutation)
    expected = attention(x, positions=positions)
    actual = attention(
        x[:, permutation], positions=positions[:, permutation]
    )[:, inverse]
    torch.testing.assert_close(actual, expected, atol=2e-6, rtol=2e-6)

test_apply_masks_preserves_effective_batch()
test_mask_ratios_do_not_collapse_across_the_effective_batch()
test_position_aware_rotary_attention_is_permutation_equivariant()

In [ ]:
#| export
class Encoder(nn.Module):
    def __init__(
        self,
        c_in,
        num_patches,
        patch_size,
        patch_stride,
        d_model,
        nhead,
        num_layers,
        use_tst_block=False,
        shared_embedding=True,
        pe_type='tAPE',
        mlp_ratio=4.0,
        qkv_bias=True,
        qk_scale=None,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        norm_layer=nn.LayerNorm,
        jepa=True,
        embed_activation=nn.GELU(),
        init_std=0.02,
        tokenizer_type='simple',
        tokenizer_kwargs={}
    ):

        super().__init__()

        # Parameters
        self.c_in = c_in
        self.patch_size = patch_size
        self.patch_stride = patch_stride
        self.d_model = d_model
        self.num_patches = num_patches
        self.activation = embed_activation if embed_activation else nn.GELU()
        self.init_std = init_std
        self.pe_type = pe_type.lower()
        self.tokenizer_type = tokenizer_type.lower()
        self.shared_embedding = shared_embedding
        self.use_tst_block = use_tst_block

        self.patch_layer = Patch(patch_len=patch_size, stride=patch_stride)
        # Building the tokenizer
        if self.tokenizer_type in ['simple_conv', 'simple']:
            self.tokenizer = TS_Tokenizer(
                c_in=c_in,
                patch_size=patch_size,
                d_model=d_model * c_in if not shared_embedding else d_model,
                patch_stride=patch_stride,
                shared_embedding=self.shared_embedding
            )
        elif self.tokenizer_type in ['complex_conv', 'complex']:
            self.tokenizer = TS_Tokenizer_Complex(
                c_in=c_in,
                patch_size=patch_size,
                d_model=d_model
            )
        elif self.tokenizer_type == 'linear':
            self.tokenizer = PatchEncoder(c_in=c_in, 
                                             patch_len=patch_size, 
                                             d_model=d_model,
                                             shared_embedding=self.shared_embedding
                                             )
        elif self.tokenizer_type == 'inception':
            self.tokenizer = InceptionTokenizer(c_in=c_in, 
                                                  patch_size=patch_size,
                                                  d_model=d_model * c_in if not shared_embedding else d_model,
                                                  patch_stride=patch_stride,
                                                  shared_embedding=self.shared_embedding,
                                                  **tokenizer_kwargs
                                                  )
        else:
            raise ValueError(f"Invalid tokenizer type: {tokenizer_type}. Valid options are: 'simple_conv', 'complex_conv', 'linear'")

        # Positional Encoder -- We use a Sin-Cos one (not learnable)
        if self.pe_type == 'tape':
            self.pe = tAPE(d_model=self.d_model, seq_len=self.num_patches)
        elif self.pe_type == 'learned':
            self.pe = PositionalEncoding(num_patch=self.num_patches, d_model=self.d_model)
        elif self.pe_type == 'rotary':
            self.pe = nn.Identity()
        
        if self.use_tst_block:
            self.dropout = nn.Dropout(drop_rate) # residual dropout
        else:
            self.dropout = nn.Identity()

        # Transformer part of the encoder
        if not use_tst_block:
            self.predictor_blocks = nn.ModuleList(
                [
                    JEPABlock(
                        dim=self.d_model,
                        num_heads=nhead,
                        mlp_ratio=mlp_ratio,
                        qkv_bias=qkv_bias,
                        qk_scale=qk_scale,
                        drop=drop_rate,
                        attn_drop=attn_drop_rate,
                        act_layer=nn.GELU,
                        norm_layer=norm_layer,
                        rotary_pes=self.pe_type == 'rotary'
                    )
                    for i in range(num_layers)
                ]
            )
        else:
            tst_block = PositionAwareTSTBlock if self.pe_type == 'rotary' else TSTBlock
            self.predictor_blocks = nn.ModuleList([tst_block(d_model=self.d_model,
                                                n_heads=nhead, 
                                                d_ff=int(self.d_model * mlp_ratio), 
                                                attn_dropout=attn_drop_rate, 
                                                dropout=drop_rate, 
                                                bias=qkv_bias,
                                                activation='gelu', 
                                                pre_norm=False, 
                                                rotary_pes=self.pe_type == 'rotary') for _ in range(num_layers)])

        if not use_tst_block:
            self.encoder_norm = nn.LayerNorm(self.d_model)
        else:
            self.encoder_norm = nn.Identity()
        self.jepa = jepa
        self.apply(self._init_weights)
        self._rescale_blocks()

    def forward(self, x, mask=None):
        # Embed the data using the Tokenizer
        bs = x.size(0)

        if self.tokenizer_type == 'linear':
            x = self.patch_layer(x, constant_pad=True, constant_pad_value=0) 
        x = self.tokenizer(x) # [bs x num_patches x (?c_in) x d_model]

        if x.dim() == 3:
            x = x.unsqueeze(2) # z: [bs x num_patch x 1 x d_model]
        x = x.transpose(1,2)
        transformer_c_in = x.size(1)
        x = torch.reshape(x, (bs * transformer_c_in, self.num_patches, self.d_model)) # u: [bs * nvars x num_patch x d_model]
        positions = torch.arange(self.num_patches, device=x.device).expand(x.size(0), -1)
        x = self.pe(x)
        x = self.dropout(x)
        # Apply mask -- In the encoder, we keep only the unmasked part
        if mask is not None and self.jepa:
            x = apply_masks(x, mask)
            positions = apply_position_masks(positions, mask)
        # Encode using Attention
        for blk in self.predictor_blocks:
            if self.pe_type == 'rotary':
                x = blk(x, mask=None, positions=positions)
            else:
                x = blk(x, mask=None)
       
        x = self.encoder_norm(x)
        return x
    
    def _rescale_blocks(self):
        def rescale(param, layer_id):
            param.div_(math.sqrt(2.0 * layer_id))
        if not self.use_tst_block:
            for layer_id, layer in enumerate(self.predictor_blocks):
                rescale(layer.attn.proj.weight.data, layer_id + 1) # rescale the attention weights
                rescale(layer.mlp.fc2.weight.data, layer_id + 1) # rescale the feedforward weights
        else:
            for layer_id, layer in enumerate(self.predictor_blocks):
                rescale(layer.self_attn.proj.weight.data, layer_id + 1) # rescale the attention weights
                rescale(layer.ff[3].weight.data, layer_id + 1) # rescale the feedforward weights

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv1d) or isinstance(m, nn.Conv2d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

In [ ]:
#| export
class Predictor(nn.Module):
    def __init__(
        self,
        num_patches,
        encoder_embed_dim=128,
        predictor_embed_dim=128,
        nhead=2,
        num_layers=1,
        use_tst_block=False,
        pe_type='tAPE',
        mlp_ratio=4.0,
        qkv_bias=True,
        qk_scale=None,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        norm_layer=nn.LayerNorm,
        embed_activation=nn.GELU(),
        init_std=0.02,
        c_in_mask_tokens = 1, # number of channels in the encoder (if treating channels sep)
    ):
        super(Predictor, self).__init__()

        # Model's parameters
        self.activation = embed_activation if embed_activation else nn.GELU()
        self.predictor_embed_dim = predictor_embed_dim
        self.num_patches = num_patches
        self.init_std = init_std
        self.pe_type = pe_type.lower()
        self.use_tst_block = use_tst_block
        self.c_in_mask_tokens = c_in_mask_tokens
        # Map the Encoder's embed dim to the predictor's embed dim
        self.predictor_embed = nn.Linear(
            encoder_embed_dim, predictor_embed_dim, bias=True
        )

        # Positional Encoder -- Note that we are using a Sin-Cos PE
        if self.pe_type == 'tape':
            self.pos_embed = nn.Parameter(
               torch.zeros(1, self.num_patches, self.predictor_embed_dim), requires_grad=False
            )
            self.init_tape_pe()
        elif self.pe_type == 'learned':
            # interpolated PEs
            n_learned_pes = min(2048, self.num_patches//4)
            self.pos_embed =  nn.Parameter(torch.empty((n_learned_pes, predictor_embed_dim)))
            nn.init.uniform_(self.pos_embed, -0.02, 0.02)
        elif self.pe_type in ['rotary', 'none']:
            self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, self.predictor_embed_dim), requires_grad=False)
        else:
            self.pos_embed = nn.Parameter(
               torch.zeros(1, self.num_patches, self.predictor_embed_dim), requires_grad=False
            )
            self.init_embed()

        if use_tst_block:
            self.dropout = nn.Dropout(drop_rate) # residual dropout
        else:
            self.dropout = nn.Identity()

        # Mask tokens
        self.mask_token = nn.Parameter(torch.zeros(c_in_mask_tokens, 1, predictor_embed_dim), requires_grad=True)
        self.mask_token = trunc_normal_(self.mask_token, std=init_std)

        # Transformer part of the Decoder
        if not use_tst_block:
            self.predictor_blocks = nn.ModuleList(
                [
                    JEPABlock(
                        dim=predictor_embed_dim,
                        num_heads=nhead,
                        mlp_ratio=mlp_ratio,
                        qkv_bias=qkv_bias,
                        qk_scale=qk_scale,
                        drop=drop_rate,
                        attn_drop=attn_drop_rate,
                        act_layer=nn.GELU,
                        norm_layer=norm_layer,
                        rotary_pes = self.pe_type == 'rotary'
                    )
                    for i in range(num_layers)
                ]
            )
        else:
            tst_block = PositionAwareTSTBlock if self.pe_type == 'rotary' else TSTBlock
            self.predictor_blocks = nn.ModuleList([tst_block(d_model=predictor_embed_dim,
                                                n_heads=nhead, 
                                                d_ff=int(predictor_embed_dim * mlp_ratio),
                                                attn_dropout=attn_drop_rate, 
                                                dropout=drop_rate, 
                                                bias=qkv_bias,
                                                activation='gelu', 
                                                pre_norm=False, 
                                                rotary_pes=self.pe_type == 'rotary') for _ in range(num_layers)])

        # To Normalize and map back to the encoder dimension (before applying
        # the loss function)
        if not use_tst_block:
            self.predictor_norm = nn.LayerNorm(predictor_embed_dim)
        else:
            self.predictor_norm = nn.Identity()
        self.predictor_proj = nn.Linear(
            predictor_embed_dim, encoder_embed_dim, bias=True
        )

        self.apply(self._init_weights)
        self._rescale_blocks()

    def forward(self, encoded_vals, mask=None, non_masks=None):

        assert (mask is not None) and (encoded_vals is not None), "No input found"
        
        _, ctx_size, _ = encoded_vals.size()
        batch_size = encoded_vals.size(0)

        # Map the output of the encoder to the Predictor's dimension
        x = self.predictor_embed(encoded_vals)

        if self.pe_type == 'learned':
            # interpolate
            pos_embed = self.pos_embed.unsqueeze(0).permute(0,2,1) # [1,d,N]
            pos_embed = F.interpolate(pos_embed, size=self.num_patches, mode='linear', align_corners=False)
            pos_embed = pos_embed.permute(0,2,1) # 1, N , d
        else:
            pos_embed = self.pos_embed

        # Add PE and apply mask to keep only the non-masked part
        cnt_pos_enc = pos_embed.repeat(batch_size, 1, 1)
        
        cnt_pos_enc = apply_masks(cnt_pos_enc, non_masks)
        x = x + cnt_pos_enc
        x = self.dropout(x)

        # Create the Target vectors and add PE
        target_pos_enc = pos_embed.repeat(batch_size, 1, 1)
        
        target_pos_enc = apply_masks(target_pos_enc, mask)
        pred_tokens = self.mask_token.repeat(batch_size // self.c_in_mask_tokens, target_pos_enc.size(1), 1)
        pred_tokens = pred_tokens + target_pos_enc
        pred_tokens = self.dropout(pred_tokens)
        # Concat the context (from the encoder) and the mask tokens
        
        x = torch.cat([x, pred_tokens], dim=1)
        positions = torch.cat([non_masks, mask], dim=1).to(x.device)
        shuffled_ = torch.randperm(x.shape[1], device=x.device) # random permutation of patch indices
        ids_restore = torch.argsort(shuffled_) # restore indices
        x = x [:,shuffled_,:] # shuffle x
        positions = positions[:, shuffled_]
        # Push through attention
        for blk in self.predictor_blocks:
            if self.pe_type == 'rotary':
                x = blk(x, mask=None, positions=positions)
            else:
                x = blk(x, mask=None)

        x = self.predictor_norm(x)

        # Output only the part related to the masked area and adapt the dim
        x = x[:, ids_restore] # restore order
        x = x[:, ctx_size:] # grab targets
        assert mask.shape[1] == x.shape[1], "The target mask shape does not equal the final shape"
        x = self.predictor_proj(x)

        return x
    
    def _rescale_blocks(self):
        def rescale(param, layer_id):
            param.div_(math.sqrt(2.0 * layer_id))
        if not self.use_tst_block:
            for layer_id, layer in enumerate(self.predictor_blocks):
                rescale(layer.attn.proj.weight.data, layer_id + 1) # rescale the attention weights
                rescale(layer.mlp.fc2.weight.data, layer_id + 1) # rescale the feedforward weights
        else:
            for layer_id, layer in enumerate(self.predictor_blocks):
                rescale(layer.self_attn.proj.weight.data, layer_id + 1) # rescale the attention weights
                rescale(layer.ff[3].weight.data, layer_id + 1) # rescale the feedforward weights

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0) 

    def init_embed(self):
        """
        This function serves for the positional encoder which is based on a
        Sin-Cos Pos Encoder.
        ---
        Users can choose any other Positional Encoder that they may see fit.
        """
        assert self.predictor_embed_dim % 2 == 0

        omega = np.arange(self.predictor_embed_dim // 2, dtype=float)
        omega /= self.predictor_embed_dim / 2.0
        omega = 1.0 / 10000**omega

        pos = np.arange(self.num_patches, dtype=float)
        pos = pos.reshape(-1)
        out = np.einsum("m,d->md", pos, omega)

        emb_sin = np.sin(out)
        emb_cos = np.cos(out)

        emb = np.concatenate([emb_sin, emb_cos], axis=1)

        self.pos_embed.data.copy_(torch.from_numpy(emb).float().unsqueeze(0))

        return emb
    
    def init_tape_pe(self):
        """
        This function serves for the positional encoder which is based on a
        Sin-Cos Pos Encoder.
        ---
        Users can choose any other Positional Encoder that they may see fit.
        """
        assert self.predictor_embed_dim % 2 == 0
        pos = torch.arange(0, self.num_patches, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, self.predictor_embed_dim, 2).float() * (-np.log(10000.0) / self.predictor_embed_dim))
        W_pos = torch.zeros(self.num_patches, self.predictor_embed_dim)
        W_pos[:, 0::2] = torch.sin((pos * div_term)*(self.predictor_embed_dim/self.num_patches)) # this is the difference between normal PE and tAPE, scaling (d_model/seq_len)
        W_pos[:, 1::2] = torch.cos((pos * div_term)*(self.predictor_embed_dim/self.num_patches))

        self.pos_embed.data.copy_(W_pos.unsqueeze(0))

        return W_pos
   
def variance_loss(x):
    return torch.mean(F.relu(1.0 - x.std(dim=1)).mean())

def loss_pred(pred, target_ema, representations=None, alpha = 0.2):
    loss = 0.0
    for pred_i, target_ema_i in zip(pred, target_ema):
        loss = loss + F.mse_loss(pred_i, target_ema_i, reduction='mean')
    loss /= len(pred)
    return loss

def mse_variance_loss(pred, target_ema, representations, alpha = 0.2):
    loss = 0.0
    for pred_i, target_ema_i, representations_i in zip(pred, target_ema, representations):
        loss = loss + F.mse_loss(pred_i, target_ema_i, reduction='mean')
        loss = loss + alpha * F.relu(1.0 - representations_i.std(dim=-1)).mean()
    loss /= len(pred)
    return loss

In [ ]:
#| export
class JEPASimpleLightning(pl.LightningModule):
    def __init__(self,
                 learning_rate,
                 train_size,
                 batch_size,
                 n_gpus,
                 patchtsjepa_encoder_kwargs,
                 patchtsjepa_predictor_kwargs,
                 weight_decay=0.04,
                 use_weight_decay_scheduler=False,
                 final_weight_decay=0.4,
                 epochs=100,
                 optimizer_type='adamw',
                 scheduler_type='OneCycle',
                 target_mask_range=(0.05,0.3), # the target can be up to 50% of the original x 
                 context_mask_range=(0.5, 1.), # the context can be up to 80% of masked out target (1-target_mask_ratio)
                 mask_block_range=(1, 30),
                 ema_decay=0.996,
                 scheduler_kwargs={},
                 transforms=None,
                 loss_fn=loss_pred
                 ):
        super().__init__()
        self.scheduler_type = scheduler_type.lower()
        self.scheduler_kwargs = scheduler_kwargs
        if self.scheduler_type is not None:
            assert self.scheduler_type in ['onecycle', 'cosineannealingwarmrestarts'], "scheduler must be either OneCycle, CosineAnnealingWarmRestarts, or None"
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.epochs = epochs
        self.optimizer_type = optimizer_type.lower()
        self.weight_decay = weight_decay
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.final_weight_decay = final_weight_decay
        self.loss_fn = loss_fn
        self.target_mask_range = target_mask_range
        self.context_mask_range = context_mask_range
        self.ema_decay = ema_decay
        self.ipe = self.train_size//self.batch_size
        self.total_steps = int(self.ipe*self.epochs)
        self.encoder = Encoder(**patchtsjepa_encoder_kwargs)
        self.num_patch = self.encoder.num_patches
        self.patch_size = self.encoder.patch_size
        self.patch_stride = self.encoder.patch_stride
        self.mask_block_range = mask_block_range
        self.c_in = self.encoder.c_in
        self.d_model = self.encoder.d_model
        patchtsjepa_predictor_kwargs['num_patches'] = self.num_patch
        self.predictor = Predictor(**patchtsjepa_predictor_kwargs)
        self.target_encoder = copy.deepcopy(self.encoder).requires_grad_(False).eval() # deterministic EMA targets
        self.transforms = transforms
        self.tokenizer_type = patchtsjepa_encoder_kwargs.get('tokenizer_type')
        self.melt_channels_to_batch = (not patchtsjepa_encoder_kwargs.get('shared_embedding') and self.tokenizer_type in ['inception', 'simple', 'simple_conv']) or (self.tokenizer_type == 'linear')
        
    def train(self, mode=True):
        """Train the online modules while keeping the EMA target deterministic."""
        super().train(mode)
        self.target_encoder.eval()
        return self

    def get_momentum_value(self):
        """Calculate momentum value based on current training step"""
        current_step = self.global_step  # PyTorch Lightning tracks this automatically
        # Ensure we don't exceed maximum momentum
        progress = min(current_step / self.total_steps, 1.0)
        # Linear warmup from ema_decay to 1.0
        momentum = self.ema_decay + progress * (1.0 - self.ema_decay)
        return momentum
    

    def ema_update(self, context_encoder, target_encoder):
        with torch.no_grad():
            m = self.get_momentum_value()
            for param_q, param_k in zip(context_encoder.parameters(), target_encoder.parameters()):
                param_k.data.mul_(m).add_((1.-m) * param_q.detach().data)
                param_k.requires_grad_(False)
        return target_encoder
    
    def forward(self, x, channel_mask=None):
        """
        should output: [bs x nvars x d_model x num_patch] 
        """
        # use context for forward
        x = self.encoder(x) # bs x n_patch x d_model
        if self.melt_channels_to_batch:
            bs = x.size(0) // self.c_in
            x = x.reshape(bs, self.c_in, -1, self.d_model)
        else:
            x = x.unsqueeze(1) # add back channel dim
        x = x.permute(0,1,3,2) # bs x nvars x d_model x n_patch, 
        return x

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        if self.transforms is not None:
            batch = self.transforms(batch)
        x, _ = batch
        bs = x.size(0)
        
        masks, non_masks = create_masks(x, patch_size=self.patch_size, patch_stride=self.patch_stride, context_mask_range=self.context_mask_range, target_mask_range=self.target_mask_range, melt_channels_to_batch=self.melt_channels_to_batch)
        masks = masks.to(x.device)
        non_masks = non_masks.to(x.device)
          
        # Predict targets
        with torch.no_grad():
            target_ema = self.target_encoder(x)
            target_ema = F.layer_norm(
                target_ema, (target_ema.size(-1),)
            )  # normalize over feature-dim  [B, N, D]
            target_ema = apply_masks(target_ema, masks)
        
        # Encode and Predict the masked tokens
        tokens = self.encoder(x, mask=non_masks)

        pred = self.predictor(tokens, mask=masks, non_masks=non_masks)

        # Compute the loss
        loss = self.loss_fn(pred, target_ema, representations=tokens, alpha=0.2)

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Native JEPA train loss is not finite at batch {batch_idx}: {loss}"
            )
        
        if batch_idx % 50 == 0:
            with torch.no_grad():
                # Check representation stats
                tokens = tokens.clone().detach()
                target_ema = target_ema.clone().detach()
                pred = pred.clone().detach()

                context_mean = tokens.mean().item()
                target_mean = target_ema.mean().item()
                target_std = target_ema.std().item()
                pred_mean = pred.mean().item()
                pred_std = pred.std().item()

                num_target_masks = torch.numel(masks)
                num_context_masks = torch.numel(non_masks)

                tokens = torch.reshape(tokens, (bs, self.c_in if self.melt_channels_to_batch else 1, -1, self.d_model)) # z: [bs x nvars x num_patch x d_model]
                target_ema = torch.reshape(target_ema, (bs, self.c_in if self.melt_channels_to_batch else 1, -1, self.d_model)) # z: [bs x nvars x num_patch x d_model]
                pred = torch.reshape(pred, (bs, self.c_in if self.melt_channels_to_batch else 1, -1, self.d_model)) # z: [bs x nvars x num_patch x d_model]

                for i, (mean, std, pred_pairwise_cos_sim) in enumerate(
                    representation_channel_stats(pred)
                ):
                    self.log(f'pred_cos_sim_channel_{i}', pred_pairwise_cos_sim)
                    self.log(f'pred_mean_channel_{i}', mean)
                    self.log(f'pred_std_channel_{i}', std)

                for i, (mean, std, context_pairwise_cos_sim) in enumerate(
                    representation_channel_stats(tokens)
                ):
                    self.log(f'context_cos_sim_channel_{i}', context_pairwise_cos_sim)
                    self.log(f'context_mean_channel_{i}', mean)
                    self.log(f'context_std_channel_{i}', std)

                for i, (mean, std, target_pairwise_cos_sim) in enumerate(
                    representation_channel_stats(target_ema)
                ):
                    self.log(f'target_cos_sim_channel_{i}', target_pairwise_cos_sim)
                    self.log(f'target_mean_channel_{i}', mean)
                    self.log(f'target_std_channel_{i}', std)

                self.log('context_mean', context_mean)
                self.log('target_mean', target_mean)
                self.log('target_std', target_std)
                self.log('pred_mean', pred_mean)
                self.log('pred_std', pred_std)
                self.log('num_target_masks', num_target_masks)
                self.log('num_context_masks', num_context_masks)

        loss = loss.to(self.device)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        return loss
    
    def on_train_batch_end(self, outputs, batch, batch_idx):
        """Called after each training batch ends"""
        # Update target encoder weights using EMA
        self.target_encoder = self.ema_update(
            self.encoder, 
            self.target_encoder
        )

    def on_train_batch_start(self, batch, batch_idx):
        # update weight decay
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd
    
    def validation_step(self, batch, batch_idx):
        x, _ = batch
        masks, non_masks = create_masks(x, patch_size=self.patch_size, patch_stride=self.patch_stride, context_mask_range=self.context_mask_range, target_mask_range=self.target_mask_range, melt_channels_to_batch=self.melt_channels_to_batch)
        masks = masks.to(x.device)
        non_masks = non_masks.to(x.device)
          
        # Predict targets
        with torch.no_grad():
            target_ema = self.target_encoder(x)
            target_ema = F.layer_norm(
                target_ema, (target_ema.size(-1),)
            )  # normalize over feature-dim  [B, N, D]
            target_ema = apply_masks(target_ema, masks)
        
        # Encode and Predict the masked tokens
        tokens = self.encoder(x, mask=non_masks)

        pred = self.predictor(tokens, mask=masks, non_masks=non_masks)

        # Compute the loss
        loss = self.loss_fn(pred, target_ema, representations=tokens, alpha=0.2)

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Native JEPA validation loss is not finite at batch {batch_idx}: {loss}"
            )

        loss = loss.to(self.device)

        self.log("val_loss", loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)

    def configure_optimizers(self):
        param_groups = [ # exclude bias and layer norm parameters from weight decay
            {
                'params': (p for n, p in self.encoder.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.predictor.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.encoder.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            }, {
                'params': (p for n, p in self.predictor.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            },
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0.0, fused=False) if self.optimizer_type == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0.0, fused=False)
        if self.scheduler_type == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, epochs=self.epochs, steps_per_epoch=self.ipe, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs) # T_0=self.epochs//10, T_mult=2, eta_min=1e-8) # lr max is initial LR
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

## Contrastive JEPA

> JEPA with cross-patient contrastive loss for patient-agnostic representations.
> Adds an InfoNCE loss on mean-pooled encoder embeddings that treats cross-patient
> samples as positives and same-patient samples as hard negatives.

In [ ]:
#| export
def cross_patient_infonce(projections, patient_ids, temperature=0.1):
    """
    InfoNCE loss that pulls cross-patient samples together and pushes same-patient
    samples apart. For each anchor, positives are all samples from *different*
    patients, and negatives are samples from the *same* patient.

    This inverts the typical contrastive objective: instead of pulling augmented
    views of the same sample together, we pull samples from different patients
    together to discourage patient-identity encoding.

    Args:
        projections: L2-normalized embeddings [B, D]
        patient_ids: integer patient IDs [B]
        temperature: softmax temperature (lower = sharper)

    Returns:
        Scalar loss value
    """
    B = projections.shape[0]
    device = projections.device

    # Cosine similarity matrix (projections are already L2-normalized)
    sim_matrix = projections @ projections.T  # [B, B]
    sim_matrix = sim_matrix / temperature

    # Build same-patient mask: True where patient_ids[i] == patient_ids[j]
    pid = patient_ids.unsqueeze(1)  # [B, 1]
    same_patient = (pid == pid.T)  # [B, B]

    # Different-patient mask (positives): True where patients differ
    diff_patient = ~same_patient  # [B, B]

    # Self-mask: exclude diagonal
    self_mask = torch.eye(B, dtype=torch.bool, device=device)
    diff_patient = diff_patient & ~self_mask

    # Check that each sample has at least one positive
    has_positives = diff_patient.any(dim=1)  # [B]
    if not has_positives.all():
        # If some samples have no cross-patient partner (rare with batch_size=128
        # and 2016 subjects), skip those in the loss
        if not has_positives.any():
            return torch.tensor(0.0, device=device, requires_grad=True)

    # For numerical stability, subtract max per row
    sim_max, _ = sim_matrix.max(dim=1, keepdim=True)
    logits = sim_matrix - sim_max.detach()

    # Denominator: exp(sim) for all pairs except self
    exp_logits = torch.exp(logits)
    exp_logits = exp_logits * (~self_mask).float()  # zero out diagonal
    denominator = exp_logits.sum(dim=1, keepdim=True)  # [B, 1]

    # Log probabilities for positive pairs
    log_prob = logits - torch.log(denominator + 1e-8)  # [B, B]

    # Average log prob over positive pairs (cross-patient) for each anchor
    num_positives = diff_patient.float().sum(dim=1)  # [B]
    # Mask out rows with no positives
    valid = num_positives > 0
    mean_log_prob_pos = (log_prob * diff_patient.float()).sum(dim=1)  # [B]
    mean_log_prob_pos = mean_log_prob_pos[valid] / num_positives[valid]

    loss = -mean_log_prob_pos.mean()
    return loss


class ContrastiveJEPALightning(pl.LightningModule):
    """
    JEPA with cross-patient contrastive loss for patient-agnostic representations.

    Combines:
    1. Standard JEPA prediction loss (MSE on masked target tokens)
    2. Cross-patient InfoNCE loss on mean-pooled encoder embeddings

    The contrastive loss operates on a projection of the mean-pooled encoder
    output (not the predictor output). It treats all cross-patient samples in
    the batch as positives and same-patient samples as hard negatives, directly
    discouraging patient-identity encoding in the representation space.

    The training_step expects batches of (X, patient_ids) where patient_ids is
    an integer tensor identifying which patient each sample belongs to.
    """

    def __init__(self,
                 learning_rate,
                 train_size,
                 batch_size,
                 n_gpus,
                 patchtsjepa_encoder_kwargs,
                 patchtsjepa_predictor_kwargs,
                 weight_decay=0.04,
                 use_weight_decay_scheduler=False,
                 final_weight_decay=0.4,
                 epochs=100,
                 optimizer_type='adamw',
                 scheduler_type='OneCycle',
                 target_mask_range=(0.05, 0.3),
                 context_mask_range=(0.5, 1.),
                 mask_block_range=(1, 30),
                 ema_decay=0.996,
                 scheduler_kwargs={},
                 transforms=None,
                 loss_fn=loss_pred,
                 # Contrastive-specific parameters
                 contrastive_weight=0.1,
                 contrastive_temperature=0.1,
                 projection_dim=128,
                 projection_hidden_dim=None,
                 ):
        super().__init__()
        self.scheduler_type = scheduler_type.lower() if scheduler_type else None
        self.scheduler_kwargs = scheduler_kwargs
        if self.scheduler_type is not None:
            assert self.scheduler_type in ['onecycle', 'cosineannealingwarmrestarts'], \
                "scheduler must be either OneCycle, CosineAnnealingWarmRestarts, or None"
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.epochs = epochs
        self.optimizer_type = optimizer_type.lower()
        self.weight_decay = weight_decay
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.final_weight_decay = final_weight_decay
        self.loss_fn = loss_fn
        self.target_mask_range = target_mask_range
        self.context_mask_range = context_mask_range
        self.ema_decay = ema_decay
        self.ipe = self.train_size // self.batch_size
        self.total_steps = int(self.ipe * self.epochs)

        # Contrastive parameters
        self.contrastive_weight = contrastive_weight
        self.contrastive_temperature = contrastive_temperature

        # Build encoder
        self.encoder = Encoder(**patchtsjepa_encoder_kwargs)
        self.num_patch = self.encoder.num_patches
        self.patch_size = self.encoder.patch_size
        self.patch_stride = self.encoder.patch_stride
        self.mask_block_range = mask_block_range
        self.c_in = self.encoder.c_in
        self.d_model = self.encoder.d_model

        # Build predictor
        patchtsjepa_predictor_kwargs['num_patches'] = self.num_patch
        self.predictor = Predictor(**patchtsjepa_predictor_kwargs)

        # EMA target encoder
        self.target_encoder = copy.deepcopy(self.encoder).requires_grad_(False).eval()

        # Projection head for contrastive loss
        # MLP: d_model → hidden → projection_dim, with BN + ReLU
        proj_hidden = projection_hidden_dim or self.d_model
        self.projection_head = nn.Sequential(
            nn.Linear(self.d_model, proj_hidden),
            nn.BatchNorm1d(proj_hidden),
            nn.ReLU(inplace=True),
            nn.Linear(proj_hidden, projection_dim),
        )

        self.transforms = transforms
        self.tokenizer_type = patchtsjepa_encoder_kwargs.get('tokenizer_type')
        self.melt_channels_to_batch = (
            not patchtsjepa_encoder_kwargs.get('shared_embedding')
            and self.tokenizer_type in ['inception', 'simple', 'simple_conv']
        ) or (self.tokenizer_type == 'linear')

    def train(self, mode=True):
        """Train online modules; keep EMA target deterministic."""
        super().train(mode)
        self.target_encoder.eval()
        return self

    def get_momentum_value(self):
        """EMA momentum: linear warmup from ema_decay to 1.0."""
        current_step = self.global_step
        progress = min(current_step / max(self.total_steps, 1), 1.0)
        return self.ema_decay + progress * (1.0 - self.ema_decay)

    def ema_update(self, context_encoder, target_encoder):
        with torch.no_grad():
            m = self.get_momentum_value()
            for param_q, param_k in zip(context_encoder.parameters(),
                                        target_encoder.parameters()):
                param_k.data.mul_(m).add_((1. - m) * param_q.detach().data)
                param_k.requires_grad_(False)
        return target_encoder

    def forward(self, x, channel_mask=None):
        """
        Inference: encode without masking.
        Returns: [bs x nvars x d_model x num_patch]
        """
        x = self.encoder(x)
        if self.melt_channels_to_batch:
            bs = x.size(0) // self.c_in
            x = x.reshape(bs, self.c_in, -1, self.d_model)
        else:
            x = x.unsqueeze(1)
        x = x.permute(0, 1, 3, 2)  # bs x nvars x d_model x n_patch
        return x

    def _compute_contrastive_loss(self, encoder_tokens, patient_ids):
        """
        Compute cross-patient contrastive loss from encoder tokens.

        Args:
            encoder_tokens: [B*c_in, n_context_patches, d_model] encoder output
            patient_ids: [B] integer patient IDs

        Returns:
            Scalar contrastive loss
        """
        # Mean pool over patches: [B*c_in, d_model]
        pooled = encoder_tokens.mean(dim=1)

        # If channels are melted to batch, reshape and mean over channels
        if self.melt_channels_to_batch:
            bs = pooled.size(0) // self.c_in
            pooled = pooled.reshape(bs, self.c_in, self.d_model)
            pooled = pooled.mean(dim=1)  # [B, d_model]

        # Project through MLP head
        projected = self.projection_head(pooled)  # [B, projection_dim]

        # L2 normalize
        projected = F.normalize(projected, dim=1)

        # Cross-patient InfoNCE
        return cross_patient_infonce(
            projected, patient_ids, temperature=self.contrastive_temperature
        )

    def training_step(self, batch, batch_idx):
        if self.transforms is not None:
            batch = self.transforms(batch)

        # Unpack: training script provides (X, patient_ids)
        x, patient_ids = batch
        bs = x.size(0)

        # Create masks
        masks, non_masks = create_masks(
            x, patch_size=self.patch_size, patch_stride=self.patch_stride,
            context_mask_range=self.context_mask_range,
            target_mask_range=self.target_mask_range,
            melt_channels_to_batch=self.melt_channels_to_batch
        )
        masks = masks.to(x.device)
        non_masks = non_masks.to(x.device)

        # Target encoder (EMA) — no gradient
        with torch.no_grad():
            target_ema = self.target_encoder(x)
            target_ema = F.layer_norm(target_ema, (target_ema.size(-1),))
            target_ema = apply_masks(target_ema, masks)

        # Online encoder with context mask
        tokens = self.encoder(x, mask=non_masks)

        # JEPA prediction loss
        pred = self.predictor(tokens, mask=masks, non_masks=non_masks)
        jepa_loss = self.loss_fn(pred, target_ema, representations=tokens, alpha=0.2)

        # Contrastive loss on encoder tokens
        contrastive_loss = self._compute_contrastive_loss(tokens, patient_ids)

        # Combined loss
        total_loss = jepa_loss + self.contrastive_weight * contrastive_loss

        if not torch.isfinite(total_loss):
            raise FloatingPointError(
                f"Contrastive JEPA train loss is not finite at batch {batch_idx}: "
                f"jepa={jepa_loss.item():.4f}, contrastive={contrastive_loss.item():.4f}"
            )

        # Logging
        self.log('train_loss', total_loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('train_jepa_loss', jepa_loss, on_step=True, on_epoch=True, sync_dist=True)
        self.log('train_contrastive_loss', contrastive_loss, on_step=True, on_epoch=True, sync_dist=True)

        if batch_idx % 50 == 0:
            with torch.no_grad():
                # Unique patients in batch
                n_unique_patients = patient_ids.unique().numel()
                self.log('batch_unique_patients', float(n_unique_patients))
                self.log('ema_momentum', self.get_momentum_value())

        return total_loss

    def on_train_batch_end(self, outputs, batch, batch_idx):
        """Update target encoder via EMA."""
        self.target_encoder = self.ema_update(self.encoder, self.target_encoder)

    def on_train_batch_start(self, batch, batch_idx):
        """Update weight decay if scheduled."""
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = (self.final_weight_decay
                      + (self.weight_decay - self.final_weight_decay)
                      * 0.5 * (1. + math.cos(math.pi * progress)))
            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)
            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd

    def validation_step(self, batch, batch_idx):
        x, patient_ids = batch

        masks, non_masks = create_masks(
            x, patch_size=self.patch_size, patch_stride=self.patch_stride,
            context_mask_range=self.context_mask_range,
            target_mask_range=self.target_mask_range,
            melt_channels_to_batch=self.melt_channels_to_batch
        )
        masks = masks.to(x.device)
        non_masks = non_masks.to(x.device)

        with torch.no_grad():
            target_ema = self.target_encoder(x)
            target_ema = F.layer_norm(target_ema, (target_ema.size(-1),))
            target_ema = apply_masks(target_ema, masks)

        tokens = self.encoder(x, mask=non_masks)
        pred = self.predictor(tokens, mask=masks, non_masks=non_masks)

        jepa_loss = self.loss_fn(pred, target_ema, representations=tokens, alpha=0.2)
        contrastive_loss = self._compute_contrastive_loss(tokens, patient_ids)
        total_loss = jepa_loss + self.contrastive_weight * contrastive_loss

        self.log('val_loss', total_loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        self.log('val_jepa_loss', jepa_loss, on_step=True, on_epoch=True, sync_dist=True)
        self.log('val_contrastive_loss', contrastive_loss, on_step=True, on_epoch=True, sync_dist=True)

    def configure_optimizers(self):
        param_groups = [
            {'params': (p for n, p in self.encoder.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))},
            {'params': (p for n, p in self.predictor.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))},
            {'params': (p for n, p in self.projection_head.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))},
            {'params': (p for n, p in self.encoder.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
             'WD_exclude': True, 'weight_decay': 0},
            {'params': (p for n, p in self.predictor.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
             'WD_exclude': True, 'weight_decay': 0},
            {'params': (p for n, p in self.projection_head.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
             'WD_exclude': True, 'weight_decay': 0},
        ]
        wd = self.weight_decay if not self.use_weight_decay_scheduler else 0.0
        if self.optimizer_type == 'adamw':
            self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=wd, fused=False)
        else:
            self.optimizer = torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=wd, fused=False)

        if self.scheduler_type == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, epochs=self.epochs, steps_per_epoch=self.ipe, **self.scheduler_kwargs)
            return {'optimizer': self.optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'step'}}
        elif self.scheduler_type == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs)
            return {'optimizer': self.optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch'}}
        else:
            return self.optimizer


## ECG-JEPA

> Adapted from https://github.com/sehunfromdaegu/ECG_JEPA/tree/master

In [ ]:
#| export
def get_1d_sincos_pos_embed_from_grid(embed_dim, pos):
    """
    embed_dim: output dimension for each position
    pos: a list of positions to be encoded: size (M,)
    out: (M, D)
    """
    assert embed_dim % 2 == 0
    omega = np.arange(embed_dim // 2, dtype=float)
    omega /= embed_dim / 2.
    omega = 1. / 10000**omega   # (D/2,)

    pos = pos.reshape(-1)   # (M,)
    out = np.einsum('m,d->md', pos, omega)   # (M, D/2), outer product

    emb_sin = np.sin(out)  # (M, D/2)
    emb_cos = np.cos(out)  # (M, D/2)

    emb = np.concatenate([emb_sin, emb_cos], axis=1)  # (M, D)
    return emb

def get_2d_sincos_pos_embed_from_grid(embed_dim, grid):
    assert embed_dim % 2 == 0

    # use half of dimensions to encode grid_h
    emb_h = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[0])  # (H*W, D/2)
    emb_w = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[1])  # (H*W, D/2)

    emb = np.concatenate([emb_h, emb_w], axis=1)  # (H*W, D)
    return emb

def get_2d_sincos_pos_embed(embed_dim, grid_size_h, grid_size_w, cls_token=False):
    """
    grid_size_h: int of the grid height
    grid_size_w: int of the grid width
    return:
    pos_embed: [grid_size_h*grid_size_w, embed_dim] or [1+grid_size_h*grid_size_w, embed_dim] (w/ or w/o cls_token)
    """
    grid_h = np.arange(grid_size_h, dtype=float)
    grid_w = np.arange(grid_size_w, dtype=float)
    grid = np.meshgrid(grid_w, grid_h)  # here w goes first
    grid = np.stack(grid, axis=0)

    grid = grid.reshape([2, 1, grid_size_h, grid_size_w])
    pos_embed = get_2d_sincos_pos_embed_from_grid(embed_dim, grid)
    if cls_token:
        pos_embed = np.concatenate([np.zeros([1, embed_dim]), pos_embed], axis=0)
    return pos_embed

    
class Encoder_Block(nn.Module):
    def __init__(self, embed_dim=384, depth=12, num_heads=6, mlp_ratio=4., qkv_bias=False, qk_scale=None,
                 drop_rate=0., attn_drop_rate=0., drop_path_rate=0., norm_layer=nn.LayerNorm):
        super().__init__() 

        self.blocks = nn.ModuleList([
            JEPABlock(
                dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop=drop_rate, attn_drop=attn_drop_rate, 
                norm_layer=norm_layer)
            for i in range(depth)])

    def forward(self, x, pos, attention_mask=None):
        for _, block in enumerate(self.blocks):
            x = block(x + pos, attention_mask)
        return x
    
class Predictor_Block(nn.Module):
    def __init__(self, predictor_embed_dim=192, depth=4, num_heads=6, mlp_ratio=4., qkv_bias=False, qk_scale=None,
                 drop_rate=0., attn_drop_rate=0., drop_path_rate=0.):
        super().__init__()    
        self.blocks = nn.ModuleList([
            JEPABlock(
                dim=predictor_embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale,
                drop=drop_rate, attn_drop=attn_drop_rate
                )
            for i in range(depth)])
    
    def forward(self, x, pos, attention_mask=None):
        for _, block in enumerate(self.blocks):
            x = block(x + pos, attention_mask)
        return x
    

# used for target/context encoding
class MaskTransformer(nn.Module):
    def __init__(
                self, 
                d_model=384,
                num_layers=12,
                nhead=6,
                mlp_ratio=4.0,
                qkv_bias=False,
                qk_scale=None,
                drop_rate=0.0,
                attn_drop_rate=0.0,
                drop_path_rate=0.0,
                norm_layer=nn.LayerNorm,
                init_std=0.02,
                mask_scale=(0.3, .5),
                mask_type='block',
                pe_type='sincos',
                c_in=3,
                num_patches=50,
                patch_size=50,
                ):
        super().__init__()

        self.mask_scale = mask_scale
        self.init_std = init_std
        # self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, num_layers)]
        self.patch_layer = Patch(patch_len=patch_size, stride=patch_size)
        self.encoder_blocks = Encoder_Block(embed_dim=d_model, depth=num_layers, num_heads=nhead, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, qk_scale=qk_scale, drop_rate=drop_rate, attn_drop_rate=attn_drop_rate, drop_path_rate=dpr, norm_layer=norm_layer)
        self.norm = nn.LayerNorm(d_model)
        self.c=c_in
        self.p = num_patches
        self.t=patch_size
        self.embed_dim = d_model
        self.W_P = nn.Linear(self.t,d_model)
        
        if pe_type == 'learnable':
            pos_embed = torch.empty((self.c*self.p, d_model))
            nn.init.uniform_(pos_embed, -0.02, 0.02)
            self.pos_embed = nn.Parameter(pos_embed, requires_grad=True)
        elif pe_type == 'sincos':
            self.pos_embed = nn.Parameter(torch.zeros(self.c*self.p, d_model), requires_grad=False)
            pos_embed = get_2d_sincos_pos_embed(d_model,self.c,self.p)
            self.pos_embed.data.copy_(torch.from_numpy(pos_embed).float())

        self.mask_type = mask_type
        # initialize the learnable token
        # trunc_normal_(self.mask_token, std=init_std)
        self.apply(self._init_weights)
        self.fix_init_weight()

    def fix_init_weight(self):
        def rescale(param, layer_id):
            param.div_(math.sqrt(2.0 * layer_id))

        for layer_id, layer in enumerate(self.encoder_blocks.blocks):
            rescale(layer.attn.proj.weight.data, layer_id + 1)
            rescale(layer.mlp.fc2.weight.data, layer_id + 1)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=self.init_std)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv1d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def _make_rand_mask(self, mask_scale):
        mask = torch.zeros(self.p, dtype=torch.bool)
        mask_ratio = mask_scale[0] + (mask_scale[1] - mask_scale[0]) * torch.rand(1).item()
        mask_num = int(self.p * mask_ratio)
        random_indices = torch.randperm(self.p)[:mask_num]
        mask[random_indices] = True
        return mask
    
    def _make_block_mask(self, mask_scale):
        mask = torch.zeros(self.p, dtype=torch.bool)
        for i in range(4):
            mask_ratio = mask_scale[0] + (mask_scale[1] - mask_scale[0]) * torch.rand(1).item()
            mask_num = int(self.p * mask_ratio)
            block_index = torch.randperm(self.p - mask_num)[0]
            mask[block_index:block_index+mask_num] = True
        return mask
    
    def _cross_attention_mask(self):
        size = self.c * self.p

        # Create row-wise mask
        row_mask = torch.zeros((size, size))
        for i in range(self.c):
            row_mask[i*self.p:(i+1)*self.p, i*self.p:(i+1)*self.p] = 1

        # Create column-wise mask
        col_mask = torch.zeros((size, size))
        for i in range(self.p):
            col_mask[i::self.p, i::self.p] = 1

        # Combine row-wise and column-wise masks
        combined_mask = row_mask + col_mask
        # Ensure values are binary
        combined_mask = combined_mask.clamp(max=1)

        return combined_mask

    def forward(self, x, mask=None):
        '''
        x : (bs, c, p, t)
        '''
        x = self.patch_layer(x)  # (bs, c, L) -> (bs, c, p, t)
        x = x.transpose(1,2)  # (bs, c, p, t)
        bs, c, p, t = x.shape
        assert c == self.c, 'Input tensor has wrong shape'
        assert p == self.p, 'Input tensor has wrong shape'
        assert t == self.t, 'Input tensor has wrong shape'

        x = x.reshape(bs, c*p, t)

        pos_embed = self.pos_embed.unsqueeze(0).expand(x.size(0), -1, -1)

        # Generate or use provided mask   
        if mask is None:
            if self.mask_type == 'random':
                mask_idx = self._make_rand_mask(self.mask_scale)
            elif self.mask_type == 'block':
                mask_idx = self._make_block_mask(self.mask_scale)
        else:
            mask_idx = mask

        vis_idx = ~mask_idx
        vis_idx = vis_idx.repeat(c)   

        # Apply projection to input tensor
        x = self.W_P(x)  # (bs, c*p, t) -> (bs, c*p, embed_dim)

        # Apply encoder block
        attention_mask = self._cross_attention_mask().to(x.device) # (c*p, c*p)

        # Slice x and positional embeddings if mask is provided
        if mask is not None:
            x = x[:,vis_idx] # (bs, c, p, embed_dim) -> (bs, c, p', embed_dim)
            pos_embed = pos_embed[:,vis_idx]  # (bs, c, p, embed_dim) -> (bs, c, p', embed_dim)
            attention_mask = attention_mask[vis_idx][:,vis_idx]
        
        x = self.encoder_blocks(x, pos_embed, attention_mask)

        # Apply normalization if specified
        if self.norm is not None:
            x = self.norm(x)

        return x, mask_idx
    

class MaskTransformerPredictor(nn.Module):
    def __init__(
                self, 
                d_model=384,
                predictor_embed_dim=192,
                num_layers=4,
                nhead=6,
                mlp_ratio=4.0,
                qkv_bias=False,
                qk_scale=None,
                drop_rate=0.0,
                attn_drop_rate=0.0,
                drop_path_rate=0.0,
                norm_layer=nn.LayerNorm,
                init_std=0.02,  
                pe_type='sincos',
                c_in=9,
                num_patches=50,
                patch_size=50,  
                ):
        
        super().__init__()
        self.predictor_embed = nn.Linear(d_model, predictor_embed_dim, bias=True)
        self.embed_dim = d_model
        self.c = c_in
        self.p = num_patches
        self.t = patch_size
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_embed_dim))

        if pe_type == 'learnable':
            pos_embed = torch.empty((self.p, predictor_embed_dim))
            nn.init.uniform_(pos_embed, -0.02, 0.02)
            self.pos_embed = nn.Parameter(pos_embed, requires_grad=False)
        elif pe_type == 'sincos':
            self.pos_embed = nn.Parameter(torch.zeros(self.p, predictor_embed_dim), requires_grad=False)
            pos_embed = get_2d_sincos_pos_embed(predictor_embed_dim,1,self.p)
            self.pos_embed.data.copy_(torch.from_numpy(pos_embed).float())

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, num_layers)]
        self.predictor_blocks = Predictor_Block(predictor_embed_dim=predictor_embed_dim, depth=num_layers, 
                                                num_heads=nhead, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, 
                                                qk_scale=qk_scale, drop_rate=drop_rate, 
                                                attn_drop_rate=attn_drop_rate, drop_path_rate=dpr)
        
        self.predictor_norm = norm_layer(predictor_embed_dim)
        self.predictor_proj = nn.Linear(predictor_embed_dim, d_model, bias=True)

        self.init_std = init_std
        trunc_normal_(self.mask_token, std=init_std)
        self.apply(self._init_weights)
        self.fix_init_weight()


    def fix_init_weight(self):
        def rescale(param, layer_id):
            param.div_(math.sqrt(2.0 * layer_id))

        for layer_id, layer in enumerate(self.predictor_blocks.blocks):
            rescale(layer.attn.proj.weight.data, layer_id + 1)
            rescale(layer.mlp.fc2.weight.data, layer_id + 1)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=self.init_std)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv1d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)


    def forward(self, x, mask):
        num_mask = mask.sum()
        x = self.predictor_embed(x)
        bs,_,pred_dim = x.shape
        x = x.reshape(bs*self.c, -1, pred_dim) # (bs, c*p, embed_dim) -> (bs*c, p, embed_dim)
        mask_token = self.mask_token.expand(x.size(0), num_mask, pred_dim)
        x = torch.cat([x, mask_token], dim=1)

        # reorder pos
        pos = self.pos_embed

        mask = mask
        vis_idx = (~mask).nonzero(as_tuple=True)[0]
        mask_idx = mask.nonzero(as_tuple=True)[0]
        idx = torch.cat((vis_idx, mask_idx))        
        
        pos = pos[idx]
        pos = pos.unsqueeze(0).expand(x.size(0), -1, -1)

        x = self.predictor_blocks(x, pos)
        x = self.predictor_norm(x) 
        x = self.predictor_proj(x)

        x = x.reshape(bs, -1, self.embed_dim)
        return x


class ECGJEPALightning(pl.LightningModule):
    def __init__(
                self, 
                encoder_kwargs,
                predictor_kwargs,
                learning_rate,
                train_size,
                batch_size,
                n_gpus,
                weight_decay=0.04,
                use_weight_decay_scheduler=False,
                final_weight_decay=0.4,
                epochs=100,
                optimizer_type='adamw',
                scheduler_type='OneCycle',
                ema_decay=0.996,
                scheduler_kwargs={},
                transforms=None,
                ):
        
        super().__init__()
        self.learning_rate = learning_rate
        self.train_size = train_size
        self.batch_size = batch_size
        self.n_gpus = n_gpus
        self.weight_decay = weight_decay
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.final_weight_decay = final_weight_decay
        self.epochs = epochs
        self.optimizer_type = optimizer_type.lower()
        self.scheduler_type = scheduler_type.lower()
        self.scheduler_kwargs = scheduler_kwargs
        self.ema_decay = ema_decay
        self.transforms = transforms
        self.encoder = MaskTransformer(**encoder_kwargs)
        self.c = encoder_kwargs['c_in']
        self.p = encoder_kwargs['num_patches']
        self.d_model = encoder_kwargs['d_model']
        self.ipe = self.train_size//self.batch_size
        self.total_steps = int(self.ipe*self.epochs)

        self.target_encoder = copy.deepcopy(self.encoder).requires_grad_(False) # no grad, ema weight updates

        self.predictor = MaskTransformerPredictor(**predictor_kwargs)
        self.loss_fn = nn.SmoothL1Loss()
        self.save_hyperparameters()
    
    def get_momentum_value(self):
        """Calculate momentum value based on current training step"""
        current_step = self.global_step  # PyTorch Lightning tracks this automatically
        # Ensure we don't exceed maximum momentum
        progress = min(current_step / self.total_steps, 1.0)
        # Linear warmup from ema_decay to 1.0
        momentum = self.ema_decay + progress * (1.0 - self.ema_decay)
        return momentum
    

    def ema_update(self, context_encoder, target_encoder):
        with torch.no_grad():
            m = self.get_momentum_value()
            for param_q, param_k in zip(context_encoder.parameters(), target_encoder.parameters()):
                param_k.data.mul_(m).add_((1.-m) * param_q.detach().data)
                param_k.requires_grad_(False)
        return target_encoder

    def forward(self, x):

        x, _ = self.encoder(x) # bs x c*p x d_model
        return x
    
    def training_step(self, batch, batch_idx):
        if self.transforms is not None:
            batch = self.transforms(batch)
        x, _ = batch
        bs = x.size(0)
        # target process
        with torch.no_grad():
            h, mask = self.target_encoder(x) # x: (bs,c, p,t), h: (bs,c*p,embed_dim)
            h = torch.nn.functional.layer_norm(h, (h.size(-1),))  # normalize over feature-dimension   
            h = h.reshape(bs, self.c, self.p, -1) # (bs,c,p,embed_dim)
            masked_h = h[:,:,mask,:]
            masked_h = masked_h.reshape(bs, -1, h.size(-1))

        # context process
        x, _ = self.encoder(x, mask) # (bs,c,p,t) -> (bs,c*p,embed_dim)
        z = self.predictor(x, mask) # (bs,c*p,embed_dim)->(bs,c*p,proj_dim)
           
        # slicing
        num_mask = mask.sum()
        z_pred = z.reshape(bs, self.c, self.p, -1)[:,:,-num_mask:,:]
        z_pred = z_pred.reshape(bs, -1, z.size(-1))

        loss = self.loss_fn(z_pred, masked_h)
        loss = loss.to(self.device)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, _ = batch
        bs = x.size(0)

        with torch.no_grad():
            h, mask = self.target_encoder(x) # x: (bs,c, p,t), h: (bs,c*p,embed_dim)
            h = torch.nn.functional.layer_norm(h, (h.size(-1),))  # normalize over feature-dimension   
            h = h.reshape(bs, self.c, self.p, -1) # (bs,c,p,embed_dim)
            masked_h = h[:,:,mask,:]
            masked_h = masked_h.reshape(bs, -1, h.size(-1))

        # context process
        x, _ = self.encoder(x, mask) # (bs,c,p,t) -> (bs,c*p,embed_dim)
        z = self.predictor(x, mask) # (bs,c*p,embed_dim)->(bs,c*p,proj_dim)
           
        # slicing
        num_mask = mask.sum()
        z_pred = z.reshape(bs, self.c, self.p, -1)[:,:,-num_mask:,:]
        z_pred = z_pred.reshape(bs, -1, z.size(-1))

        loss = self.loss_fn(z_pred, masked_h)
        loss = loss.to(self.device)

        self.log("val_loss", loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)

    def on_train_batch_end(self, outputs, batch, batch_idx):
        """Called after each training batch ends"""
        # Update target encoder weights using EMA
        self.target_encoder = self.ema_update(
            self.encoder, 
            self.target_encoder
        )

    def on_train_batch_start(self, batch, batch_idx):
        # update weight decay
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd
    
    def configure_optimizers(self):
        param_groups = [ # exclude bias and layer norm parameters from weight decay
            {
                'params': (p for n, p in self.encoder.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.predictor.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.encoder.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            }, {
                'params': (p for n, p in self.predictor.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            },
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0.0, fused=False) if self.optimizer_type == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0.0, fused=False)
        if self.scheduler_type == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, epochs=self.epochs, steps_per_epoch=self.ipe, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs) # T_0=self.epochs//10, T_mult=2, eta_min=1e-8) # lr max is initial LR
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()